In [38]:
import warnings
warnings.filterwarnings('ignore')
import hoomd
import gsd
import matplotlib.pyplot as plt
import numpy as np
import gsd.hoomd
from flowermd.base import Pack,Lattice, Simulation
from flowermd.library import EllipsoidForcefield, EllipsoidChain
from flowermd.utils import get_target_box_number_density
from flowermd.utils.constraints import create_rigid_ellipsoid_chain
import unyt as u
import hoomd

In [39]:
class EllipsoidChainBetter(EllipsoidChain):
    """Create an ellipsoid polymer chain.

    This is a coarse-grained molecule where each monomer is modeled
    as an anisotropic bead (i.e. ellipsoid).

    Notes
    -----
    In order to form chains of connected ellipsoids, "ghost"
    particles of types "A" and "B" are used.

    This is meant to be used with
    `flowermd.library.forcefields.EllipsoidForcefield`
    and requires using `flowermd.utils.constraints.set_bond_constraints` to set up
    the fixed bonds correctly in HOOMD-Blue.

    Parameters
    ----------
    lengths : int, required
        The number of monomer repeat units in the chain.
    num_mols : int, required
        The number of chains to create.
    lpar : float, required
        The semi-axis length of the ellipsoid bead along its major axis.
    bead_mass : float, required
        The mass of the ellipsoid bead.
    """

    def __init__(self, lengths, num_mols, lpar, bead_mass, bond_L=0.1):
        self.bead_mass = bead_mass
        self.lpar = lpar
        self.bond_L = bond_L
        # get the indices of the particles in a rigid body
        self.bead_constituents_types = ["X", "A", "T", "T"]
        super(EllipsoidChain, self).__init__(lengths=lengths, num_mols=num_mols)

    def _build(self, length):
        # Build up ellipsoid bead
        bead = mb.Compound(name="ellipsoid")
        center = mb.Compound(pos=(0, 0, 0), name="X", mass=self.bead_mass / 4)
        
        head = mb.Compound(
            pos=(self.lpar + (self.bond_L / 2), 0, 0) if length > 1 else (0, 0, 0),
            name="A",
            mass=self.bead_mass / 4 if length > 1 else 0.00001,
        )
        tether_head = mb.Compound(
            pos=(self.lpar, 0, 0), name="T", mass=self.bead_mass / 4
        )
        tether_tail = mb.Compound(
            pos=(-self.lpar, 0, 0), name="T", mass=self.bead_mass / 4
        )
        bead.add([center, head, tether_head, tether_tail])
        bead.add_bond([center, head])

        chain = mb.Compound()
        last_bead = None
        for i in range(length):
            translate_by = np.array([(i * self.lpar * 2) + self.bond_L, 0, 0])
            this_bead = mb.clone(bead)
            this_bead.translate(by=translate_by)
            chain.add(this_bead)
            if last_bead:
                chain.add_bond([this_bead.children[0], last_bead.children[1]])
                chain.add_bond([this_bead.children[3], last_bead.children[2]])
            last_bead = this_bead

        return chain

In [ ]:
ellipsoid_chain = EllipsoidChainBetter(lengths=1,num_mols=128,lpar=1.0,bead_mass=1.0)
ff = EllipsoidForcefield(epsilon=1.0,lpar=1.0,lperp=0.5,r_cut=2.0)
ff.hoomd_forces
system = Pack(molecules=ellipsoid_chain, density=0.1*u.Unit("nm**-3"), packing_expand_factor=6,edge=2,overlap=1,fix_orientation=True)
#system = Lattice(molecules=ellipsoid_chain,x=1,y=1,n=4)
gsd_path=('ellipsoid-chain-2mer.gsd')
rigid_frame, rigid = create_rigid_ellipsoid_chain(
    system.hoomd_snapshot
)
ellipsoid_sim = Simulation(
    initial_state=rigid_frame,
    forcefield=ff.hoomd_forces,
    constraint=rigid,
    dt=0.001,
    gsd_write_freq=int(1e3),
    gsd_file_name=gsd_path,
    log_write_freq=int(1e4),
    log_file_name='log.txt')

target_box = get_target_box_number_density(density=0.7*u.Unit("nm**-3"),n_beads=100)
ellipsoid_sim.run_update_volume(final_box_lengths=target_box, kT=6.0, n_steps=1e5,tau_kt=100*ellipsoid_sim.dt,period=10,thermalize_particles=True)
print("shrink finished")
ellipsoid_sim.run_NVT(n_steps=1e5, kT=1.0, tau_kt=10*ellipsoid_sim.dt)
#ellipsoid_sim.save_restart_gsd("restart.gsd")
ellipsoid_sim.flush_writers()
#ellipsoid_sim.save_simulation("sim.pickle")

Initializing simulation state from a gsd.hoomd.Frame.
Step 5500 of 100000; TPS: 5631.36; ETA: 0.3 minutes
Step 11000 of 100000; TPS: 6702.25; ETA: 0.2 minutes
Step 16500 of 100000; TPS: 7240.41; ETA: 0.2 minutes
Step 22000 of 100000; TPS: 7487.54; ETA: 0.2 minutes
Step 27500 of 100000; TPS: 7702.77; ETA: 0.2 minutes
Step 33000 of 100000; TPS: 7817.02; ETA: 0.1 minutes
Step 38500 of 100000; TPS: 7895.37; ETA: 0.1 minutes
Step 44000 of 100000; TPS: 7988.8; ETA: 0.1 minutes
Step 49500 of 100000; TPS: 8055.74; ETA: 0.1 minutes
Step 55000 of 100000; TPS: 8118.24; ETA: 0.1 minutes
Step 60500 of 100000; TPS: 8160.61; ETA: 0.1 minutes
Step 66000 of 100000; TPS: 8173.57; ETA: 0.1 minutes
Step 71500 of 100000; TPS: 8165.88; ETA: 0.1 minutes
Step 77000 of 100000; TPS: 8133.83; ETA: 0.0 minutes
Step 82500 of 100000; TPS: 8033.68; ETA: 0.0 minutes
Step 88000 of 100000; TPS: 7794.11; ETA: 0.0 minutes
Step 93500 of 100000; TPS: 7238.37; ETA: 0.0 minutes
Step 99000 of 100000; TPS: 5435.61; ETA: 0.0 mi

In [ ]:
def ellipsoid_gsd(gsd_file, new_file, ellipsoid_types, lpar, lperp):
    """Add needed information to GSD file to visualize ellipsoids.

    Saves a new GSD file with lpar and lperp values populated
    for each particle. Ovito can be used to visualize the new GSD file.

    Parameters
    ----------
    gsd_file : str
        Path to the original GSD file containing trajectory information
    new_file : str
        Path and filename of the new GSD file
    ellipsoid_types : str or list of str
        The particle types (i.e. names) of particles to be drawn
        as ellipsoids.
    lpar : float
        Value of lpar of the ellipsoids
    lperp : float
        Value of lperp of the ellipsoids

    """
    with gsd.hoomd.open(new_file, "w") as new_t:
        with gsd.hoomd.open(gsd_file) as old_t:
            for snap in old_t:
                shape_dicts_list = []
                for ptype in snap.particles.types:
                    if ptype == ellipsoid_types or ptype in ellipsoid_types:
                        shapes_dict = {
                            "type": "Ellipsoid",
                            "a": lpar,
                            "b": lperp,
                            "c": lperp,
                        }
                    else:
                        shapes_dict = {"type": "Sphere", "diameter": 0.001}
                    shape_dicts_list.append(shapes_dict)
                snap.particles.type_shapes = shape_dicts_list
                snap.validate()
                new_t.append(snap)

gsd_path=('ellipsoid-chain-2mer.gsd')
ellipsoid_gsd(gsd_file=gsd_path,new_file="ovito-ellipsoid.gsd",ellipsoid_types='R',lpar=1.0,lperp=0.5)